In [1]:
import pandas as pd
import numpy as np 
import pyarrow as pa

## 模块一

In [3]:
# 练习1
values = {"产品":("A","B","C"), "价格":(10,20,30), "销量":(100,200,150)}
df = pd.DataFrame(values)
df = df.convert_dtypes(dtype_backend="numpy_nullable")
df

,产品,价格,销量
0,A,10,100
1,B,20,200
2,C,30,150


In [10]:
# 判断是否为扩展类型 pandas extension
from pandas.api.types import is_extension_array_dtype

def classify_dtype(dtype):
    if isinstance(dtype, pd.ArrowDtype):
        return "PyArrow 类型"
    elif is_extension_array_dtype(dtype):
        return "Pandas 扩展类型"
    else:
        return "NumPy 类型"
        
for col in df.columns:
    col_dtype = df[col].dtype
    print(df[col].name, classify_dtype(col_dtype))
    

产品 Pandas 扩展类型
价格 Pandas 扩展类型
销量 Pandas 扩展类型


In [12]:
# 练习2
df.dtypes # 每列数据类型
df.shape

(3, 3)

In [15]:
# 练习3
df.iloc[[1], :]
df.loc[:, ["价格"]]

,价格
0,10
1,20
2,30


## 模块二

### 练习4

In [ ]:
df.to_csv("product.csv", index=False)

df_csv = pd.read_csv("product.csv", dtype_backend="numpy_nullable")

df_csv.dtypes

for col in df_csv.columns:
    col_dtype = df[col].dtype
    print(df[col].name, classify_dtype(col_dtype))

In [24]:
df_temp = pd.DataFrame({'id': [1, 2], 'name': ['Tom', 'Jerry']})
df_temp.to_csv('test.csv', index=False)

In [26]:
df = pd.read_csv("test.csv", dtype_backend="numpy_nullable")
df

,id,name
0,1,Tom
1,2,Jerry


### 练习5

In [28]:
df = pd.read_csv("test.csv", dtype_backend="numpy_nullable", usecols=["name"], nrows=1)
df

,name
0,Tom


### 练习6

In [29]:
df = pd.read_csv("product.csv", dtype_backend="numpy_nullable")
df

,产品,价格,销量
0,A,10,100
1,B,20,200
2,C,30,150


In [35]:
df.to_excel("product.xlsx", sheet_name="Sheet1", index=False)

In [36]:
df = pd.read_excel("product.xlsx", dtype_backend="numpy_nullable", sheet_name="Sheet1")
df

,产品,价格,销量
0,A,10,100
1,B,20,200
2,C,30,150


## 模块三

### 练习7

In [54]:
values = {'A': [1, None, 3], 'B': [None, 5, 6]}
df = pd.DataFrame(values)
df = df.convert_dtypes(dtype_backend="numpy_nullable")
df

,A,B
0,1,<NA>
1,<NA>,5
2,3,6


In [41]:
df.isnull()

,A,B
0,False,True
1,True,False
2,False,False


In [42]:
df.isnull().sum()

A    1
B    1
dtype: int64

### 练习8

In [45]:
df.dropna(axis=0)

,A,B
2,3,6


### 9

In [57]:
df["B"] = df["B"].astype(pd.Float32Dtype())
df.fillna(value={"A":0, "B":df["B"].mean()})

## 模块四

In [59]:
df = pd.read_csv("product.csv", dtype_backend="numpy_nullable")
df

,产品,价格,销量
0,A,10,100
1,B,20,200
2,C,30,150


### 10

In [62]:
mask = (df["价格"] > 15) & (df["销量"] > 150)
mask

0    False
1     True
2    False
dtype: boolean

In [63]:
df[mask]

,产品,价格,销量
1,B,20,200


In [69]:
df[[True, False, True]]

,产品,价格,销量
0,A,10,100
2,C,30,150


### 11

In [70]:
df.sort_values(by="销量", ascending=False)

,产品,价格,销量
1,B,20,200
2,C,30,150
0,A,10,100


In [72]:
df["销售额"] = df["销量"] * df["价格"]
df

,产品,价格,销量,销售额
0,A,10,100,1000
1,B,20,200,4000
2,C,30,150,4500


In [74]:
df["等级"] = df["价格"].apply(func=lambda x: "高" if x>20 else "低")
df

,产品,价格,销量,销售额,等级
0,A,10,100,1000,低
1,B,20,200,4000,低
2,C,30,150,4500,高


## 模块五

In [75]:
df = pd.DataFrame(data={'城市': ['北京','上海','北京','上海'], '销售额': [100, 200, 300, 400]})
df = df.convert_dtypes("numpy_nullable")
df

,城市,销售额
0,北京,100
1,上海,200
2,北京,300
3,上海,400


### 13

In [81]:
df.groupby(by="城市").agg(总销售额=pd.NamedAgg(column="销售额", aggfunc="sum"))

,总销售额
城市,
上海,600
北京,400


In [82]:
df.groupby("城市")["销售额"].sum()

城市
上海    600
北京    400
Name: 销售额, dtype: Int64

### 14

In [83]:
df.groupby("城市")["销售额"].agg(["sum", "min", "count"])

,sum,min,count
城市,,,
上海,600,200,2
北京,400,100,2


### 15

In [85]:
data = {
    '城市': ['北京', '上海', '北京', '上海', '广州'],
    '产品': ['A', 'B', 'C', 'A', 'B'],
    '销售额': [100, 200, np.nan, 400, 50]
}
df = pd.DataFrame(data=data)
df = df.convert_dtypes(dtype_backend="numpy_nullable")
df

,城市,产品,销售额
0,北京,A,100
1,上海,B,200
2,北京,C,<NA>
3,上海,A,400
4,广州,B,50


In [86]:
df.to_csv("sales.csv", index=False)

In [91]:
df = pd.read_csv("sales.csv", dtype_backend="numpy_nullable")
df["销售额"] = df["销售额"].astype(dtype=pd.Float32Dtype())
df

,城市,产品,销售额
0,北京,A,100.0
1,上海,B,200.0
2,北京,C,<NA>
3,上海,A,400.0
4,广州,B,50.0


In [93]:
df.fillna(value={"销售额": df["销售额"].mean()}, inplace=True)
df

,城市,产品,销售额
0,北京,A,100.0
1,上海,B,200.0
2,北京,C,187.5
3,上海,A,400.0
4,广州,B,50.0


In [95]:
df.groupby(by="城市").agg(总销售额=pd.NamedAgg(column="销售额", aggfunc="sum"))

,总销售额
城市,
上海,600.0
北京,287.5
广州,50.0


In [100]:
sum_sales = df.groupby(by="城市").agg(总销售额=pd.NamedAgg(column="销售额", aggfunc="sum"))
ser = sum_sales["总销售额"]
ser[ser>300]

城市
上海    600.0
Name: 总销售额, dtype: Float32